In [ ]:
import json
import pandas as pd

with open("../data/jobs_results_enriched.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.json_normalize(data)

print(df.shape)
df.head()

In [ ]:
print(df.columns.tolist())
print(df.dtypes)
df.isnull().sum()

In [ ]:
rows_before = len(df)

if "job_id" in df.columns and df["job_id"].notna().any():
    # Use job_id when it actually contains values.
    df = df.drop_duplicates(subset=["job_id"], keep="first").copy()
    duplicate_key_used = ["job_id"]

else:
    # Fallback key requested for datasets without job_id.
    fallback_cols = ["job_title", "company", "location"]

    missing = [col for col in fallback_cols if col not in df.columns]
    if missing:
        raise KeyError(f"Missing columns required for duplicate removal: {missing}")

    # Temporary normalized columns make duplicate matching more reliable.
    for col in fallback_cols:
        df[f"_key_{col}"] = (
            df[col]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
        )

    normalized_key = [f"_key_{col}" for col in fallback_cols]
    df = df.drop_duplicates(subset=normalized_key, keep="first").copy()
    df.drop(columns=normalized_key, inplace=True)
    duplicate_key_used = fallback_cols

rows_after = len(df)

print("Duplicate key used:", duplicate_key_used)
print("Rows before:", rows_before)
print("Rows after :", rows_after)
print("Duplicates removed:", rows_before - rows_after)

In [ ]:
# =========================
# Fix salary data type
# =========================

if "salary" in df.columns:

    # Replace missing salary values with NA
    df["salary"] = df["salary"].replace(
        ["N/A", "NA", "NONE", "NULL", ""],
        pd.NA
    )

    # Remove commas from values such as "12,000"
    df["salary"] = (
        df["salary"]
        .astype("string")
        .str.replace(",", "", regex=False)
    )

    # Convert salary from string to numeric
    df["salary"] = pd.to_numeric(
        df["salary"],
        errors="coerce"
    )

# Check the result
print("Salary data type:", df["salary"].dtype)

df[["salary"]].head(10)

In [ ]:
date_columns = [
    "posted_at",
    "fetched_at",
    "updated",
    "created_at"
]

converted_date_cols = []

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)
        converted_date_cols.append(col)

print("Converted date columns:", converted_date_cols)

if converted_date_cols:
    display(df[converted_date_cols].head())

In [ ]:
TRUE_VALUES = {"true", "yes", "1", "y", "t"}
FALSE_VALUES = {"false", "no", "0", "n", "f"}
BOOLEAN_VALUES = TRUE_VALUES | FALSE_VALUES


def convert_boolean_like_columns(dataframe):
    converted = []

    for col in dataframe.select_dtypes(include=["object", "string"]).columns:
        non_null = dataframe[col].dropna().astype(str).str.strip().str.lower()

        # Skip empty columns.
        if non_null.empty:
            continue

        unique_values = set(non_null.unique())

        # Convert only if every observed value is boolean-like.
        if unique_values.issubset(BOOLEAN_VALUES):
            mapping = {value: True for value in TRUE_VALUES}
            mapping.update({value: False for value in FALSE_VALUES})

            dataframe[col] = (
                dataframe[col]
                .astype("string")
                .str.strip()
                .str.lower()
                .map(mapping)
                .astype("boolean")
            )
            converted.append(col)

    return converted


converted_bool_cols = convert_boolean_like_columns(df)
print("Converted boolean columns:", converted_bool_cols)

In [ ]:
print("\nData types after transformation:")
print(df.dtypes)

print("\nCleaned sample:")
display(
    df[
        [
            "job_title",
            "company",
            "location",
            "salary",
            "posted_at",
            "experience_level"
        ]
    ].head(10)
)

In [ ]:
df.head(10)